# YOLO Training Template — Campus Hazard Detection

**One notebook per member.** Copy this to `train_yolo_member1.ipynb` etc.
and set `MEMBER` below. Each member trains ONE model over 5 classes
(see `config/classes.yaml`). The report (§5, §8, §13) needs the
hyperparameters, training iterations, and evaluation metrics produced here.

---
## ⚡ Google Colab users — run this cell first
Skip if running locally.
```python
# Mount your shared Drive (where the repo + dataset live)
from google.colab import drive
drive.mount('/content/drive')

# cd into the repo root on Drive (adjust path to match yours)
import os
os.chdir('/content/drive/MyDrive/Machine Learning Project')

# Install GPU torch + ultralytics (Colab already has CUDA)
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -q ultralytics
```
After mounting, skip the `download_weights.py` cell — ultralytics can auto-download on Colab.

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
# Local install order matters: GPU torch FIRST, then ultralytics.
#   pip install torch torchvision --index-url https://download.pytorch.org/whl/cu124
#   pip install ultralytics
import os, subprocess, sys, pathlib
import torch
from ultralytics import YOLO

print('CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
DEVICE = 0 if torch.cuda.is_available() else 'cpu'

# ── Repo root (works whether notebook is run from ml/notebooks/ or repo root) ─
NB_DIR = pathlib.Path(os.getcwd())          # directory Jupyter was launched from
# If launched from ml/notebooks/, walk up two levels; if from repo root, stay.
REPO_ROOT = NB_DIR if (NB_DIR / 'config').exists() else NB_DIR.parent.parent
os.chdir(REPO_ROOT)                         # make all relative paths work
print('Repo root:', REPO_ROOT)

# ── Pretrained weights ────────────────────────────────────────────────────────
# Local machines: SSL inspection blocks ultralytics auto-download → use helper.
# Colab: skip this block; ultralytics will auto-download yolov8n.pt fine.
DL_SCRIPT = REPO_ROOT / 'ml' / 'notebooks' / 'download_weights.py'
if DL_SCRIPT.exists():
    subprocess.run([sys.executable, str(DL_SCRIPT), 'n'], check=True)
WEIGHTS = str(REPO_ROOT / 'yolov8n.pt')    # saved to repo root by download_weights.py

# ── Member config ─────────────────────────────────────────────────────────────
MEMBER = 'member1'  # <-- CHANGE THIS: member1 | member2 | member3 | member4
DATA_CONFIG = str(REPO_ROOT / 'config' / f'{MEMBER}_data.yaml')
assert os.path.exists(DATA_CONFIG), f'Not found: {DATA_CONFIG}'
print('Training', MEMBER, 'from', DATA_CONFIG)

## Hyperparameters
Document every value you use — the report requires epochs, batch size, image
size, learning rate, optimizer, and augmentation settings. Record each
training *iteration* (initial run, retrain with more data, augmentation tuning)
in your logbook (`docs/logbook_template.csv`).

In [ ]:
model = YOLO(WEIGHTS)   # nano = fast + low VRAM; try yolov8s.pt for better accuracy

# 4 GB VRAM (RTX 3050): batch=8 + imgsz=640 fits fine.
# Out of memory? Lower batch to 4 or imgsz to 512.
# Colab T4 (15 GB): safe to use batch=16.
results = model.train(
    data=DATA_CONFIG,
    epochs=100,
    imgsz=640,
    batch=8,
    device=DEVICE,
    lr0=0.01,
    optimizer='auto',
    patience=20,        # early stopping — saves time if val plateaus
    project=f'runs/{MEMBER}',
    name='train',
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, mosaic=1.0,
)

## Evaluation (assignment §13)
Report precision, recall, mAP@0.5, mAP@0.5:0.95, and the confusion matrix.
Ultralytics writes the confusion matrix + PR curves to the run folder — screenshot these for the report.

In [ ]:
best_pt = str(REPO_ROOT / 'runs' / MEMBER / 'train' / 'weights' / 'best.pt')
model = YOLO(best_pt)
metrics = model.val(data=DATA_CONFIG, device=DEVICE)
print('mAP@0.5      :', round(metrics.box.map50, 4))
print('mAP@0.5:0.95 :', round(metrics.box.map,   4))
print('Precision    :', round(metrics.box.mp,     4))
print('Recall       :', round(metrics.box.mr,     4))
print()
print('Per-class mAP@0.5:')
for name, ap in zip(metrics.names.values(), metrics.box.ap50):
    print(f'  {name:<30} {ap:.4f}')
# Confusion matrix + PR curves saved under runs/{MEMBER}/train/

## Error analysis
Inspect false positives / false negatives, class confusion (especially between
overlap classes), and small-object misses. Note improvement attempts here for
the report's training-iteration table.

In [ ]:
import shutil

# Copy best.pt to models/ so the backend and meta-classifier can find it.
# Then share the file with M3 via Google Drive (git ignores *.pt).
dest = str(REPO_ROOT / 'models' / f'{MEMBER}.pt')
shutil.copy(best_pt, dest)
print(f'Copied best.pt → {dest}')
print('Next: upload this file to the shared Google Drive and tell M3.')